In [ ]:
import arcpy
import os

In [ ]:
buffer_complete = r"F:\newData\hesse_shp\boundingBox_hesse.shp"
buffer_complete_dis = r"F:\newData\hesse_shp\bbmeged_hesse.shp"

inp_fishnet = r"F:\newData\hesse_shp\hesse_fishnet_50x50.shp"

clipped = r"F:\newData\hesse_shp\hesse_raster_50x50.shp"

In [ ]:
arcpy.management.Dissolve(buffer_complete, buffer_complete_dis)

In [ ]:
arcpy.analysis.Clip(inp_fishnet, buffer_complete_dis, clipped)

In [ ]:
with arcpy.da.UpdateCursor(clipped, ["*"]) as cursor:
    i = 0
    for row in cursor:
        filename = r"F:\newData\hesse_shp\grid\hesse_raster_50x50_{}.shp".format(i)
        arcpy.management.CopyFeatures(clipped, filename, '', None, None, None)
        i = i + 1



In [ ]:
workspace = r"F:\newData\hesse_shp\grid"

datalist = []

for dirpath, dirnames, filenames in arcpy.da.Walk(workspace):
    for filename in filenames:
        datalist.append(r"{0}\{1}".format(dirpath, filename))
        #datalist.append(os.path.join(dirpath, filename))

In [ ]:
print(datalist)

In [ ]:
i = 0
for file in datalist:
    
    with arcpy.da.UpdateCursor(file, ["*"]) as cursor:
        for row in cursor:
            if row[0] == i:
                continue
            else:
                cursor.deleteRow()
    i = i+1

In [ ]:
# Input Files
inp_vect_water = r"F:\newData\hesse_shp\gis_osm_water_a_free_1.shp"
inp_vect_build = r"F:\newData\hesse_shp\gis_osm_buildings_a_free_1.shp"
inp_vect_landuse = r"F:\newData\hesse_shp\gis_osm_landuse_a_free_1.shp"
inp_vect_roads = r"F:\newData\hesse_shp\gis_osm_roads_free_1.shp"
inp_vect_railways = r"F:\newData\hesse_shp\gis_osm_railways_free_1.shp"

# import rasters to clip
diff_1914 = r"F:\newData\testworkspace.gdb\copyraster1914"
diff_2119 = r"F:\newData\testworkspace.gdb\copyraster2119_2"

# Clip analysis
#arcpy.analysis.Clip(inp_vect_water, area_of_interest, out_data_water)  
#arcpy.analysis.Clip(inp_vect_build, area_of_interest, out_data_build)  
#arcpy.analysis.Clip(inp_vect_landuse, area_of_interest, out_data_landuse)  

In [ ]:
i = 0
for file in datalist:
    out_file_w = r"F:\newData\hesse_shp\grid\osm_" + r"{}\water_{}.shp".format(i, i)
    out_file_b = r"F:\newData\hesse_shp\grid\osm_" + r"{}\build_{}.shp".format(i, i)
    out_file_l = r"F:\newData\hesse_shp\grid\osm_" + r"{}\landu_{}.shp".format(i, i)
    out_file_ro = r"F:\newData\hesse_shp\grid\osm_" + r"{}\roads_{}.shp".format(i, i)
    out_file_ra = r"F:\newData\hesse_shp\grid\osm_" + r"{}\railw_{}.shp".format(i, i)
    
    out_diff_19_14 = r"F:\newData\hesse_shp\grid\osm_" + r"{}\diff_19_14_{}.tif".format(i, i)
    out_diff_21_19 = r"F:\newData\hesse_shp\grid\osm_" + r"{}\diff_21_19_{}.tif".format(i, i)
    
    #print(out_file_w, out_file_b, out_file_l,out_file_ro, out_file_ra)
    
    arcpy.analysis.Clip(inp_vect_water, file, out_file_w)
    arcpy.analysis.Clip(inp_vect_build, file, out_file_b)
    arcpy.analysis.Clip(inp_vect_landuse, file, out_file_l)
    arcpy.analysis.Clip(inp_vect_roads, file, out_file_ro)
    arcpy.analysis.Clip(inp_vect_railways, file, out_file_ra)

    # clip and save all raster data
    diff_1914_clip = Clip(diff_1914, file) 
    diff_1914_clip.save(out_diff_19_14) 

    diff_2119_clip = Clip(diff_2119, file) 
    diff_2119_clip.save(out_diff_21_19) 
    
    i = i+1

In [ ]:
workspace = r"F:\newData\hesse_shp\grid"
for dirpath, dirnames, filenames in arcpy.da.Walk(workspace, datatype="FeatureClass"):
    for filename in filenames:
        filepath = r"{0}\{1}".format(dirpath, filename)
        out_diss = r"{0}\dis_{1}".format(dirpath, filename)  # ACHTUNG! Landuse!!!
        arcpy.management.Dissolve(filepath,out_diss)
        arcpy.management.Delete(filepath)

In [ ]:
workspace = r"F:\newData\hesse_shp\grid"

for dirpath, dirnames, filenames in arcpy.da.Walk(workspace, datatype="FeatureClass"):
    for filename in filenames:
        distance = "10 Meters"
        filepath = r"{0}\{1}".format(dirpath, filename)
        out_buf = r"{0}\buf10_{1}".format(dirpath, filename)
        arcpy.analysis.Buffer(filepath, out_buf, distance, "FULL", "ROUND", "ALL", None, "PLANAR")

In [ ]:
# Quarrys extrahieren! -> nicht getestet wegen Landuse zu früh "Dissolved"
workspace = r"F:\newData\hesse_shp\grid"
field_name = "fclass"

for dirpath, dirnames, filenames in arcpy.da.Walk(workspace):
    for filename in filenames:
        #if filename.startswith(r'buf10_dis_landu') and filename.endswith(r'{}.shp'.format(i)):
        if filename.startswith(r'buf10_dis_landu'):
            inp_filepath = r"{0}\{1}".format(dirpath, filename)
            out_filepath = r"{0}\{1}_quarry{2}".format(dirpath, filename[0:9], filename[filename.rfind('_'):])
            arcpy.management.CopyFeatures(inp_filepath, out_filepath, '', None, None, None)
            with arcpy.da.UpdateCursor(out_filepath, [field_name]) as cursor:
                for row in cursor:
                    if row[0] == 'quarry':
                        continue
                    else:
                        cursor.deleteRow()

In [ ]:
# Quarries from only_quarry.shp 

workspace = r"F:\newData\hesse_shp\grid"
distance = "10 Meters"
inp_vect_quarry = r"F:\newData\hesse_osm_buffer\only_quarry.shp"

i = 0
for dirpath, dirnames, filenames in arcpy.da.Walk(workspace):
    for filename in filenames:
        if filename.startswith(r'dis_hesse_raster_50x50'):
            inp_filepath = r"{0}\{1}".format(dirpath, filename)
            out_filepath = r"{0}\osm_{1}\{2}_quarry_{3}".format(dirpath, i, filename[4:9], i) #filename[filename.rfind('_'):]
            dis_filepath = r"{0}\osm_{1}\{2}_quarry_{3}".format(dirpath, i, filename[0:9], i)
            out_buf = r"{0}\osm_{1}\buf10_dis_quarry_{2}".format(dirpath, i, i)
            arcpy.analysis.Clip(inp_vect_quarry, inp_filepath, out_filepath)
            arcpy.management.Dissolve(out_filepath, dis_filepath)
            arcpy.analysis.Buffer(dis_filepath, out_buf, distance, "FULL", "ROUND", "ALL", None, "PLANAR")
            print(inp_vect_quarry, inp_filepath, out_filepath, dis_filepath)
            print(out_buf, inp_vect_quarry)
            i = i+1

In [ ]:
# mergen der benutzen params

workspace = r"F:\newData\hesse_shp\grid"
#i = 0
for i in range(0, 18): 
    inp_build = r"F:\newData\hesse_shp\grid\osm_{}\buf10_dis_build_{}".format(i, i)
    inp_quarry = r"F:\newData\hesse_shp\grid\osm_{}\buf10_dis_quarry_{}".format(i, i)
    inp_water = r"F:\newData\hesse_shp\grid\osm_{}\buf10_dis_water_{}".format(i, i)
    
    inp_roads = r"F:\newData\hesse_shp\grid\osm_{}\buf10_dis_roads_{}".format(i, i)
    inp_railways = r"F:\newData\hesse_shp\grid\osm_{}\buf10_dis_railw_{}".format(i, i)
    
    out_merged_3 = r"F:\newData\hesse_shp\grid\osm_{}\buf10_merged3_{}".format(i, i)
    out_dis_merged_3 = r"F:\newData\hesse_shp\grid\osm_{}\buf10_dis_merged3_{}".format(i, i)
    
    out_merged_5 = r"F:\newData\hesse_shp\grid\osm_{}\buf10_merged5_{}".format(i, i)
    out_dis_merged_5 = r"F:\newData\hesse_shp\grid\osm_{}\buf10_dis_merged5_{}".format(i, i)
    
    arcpy.management.Merge([inp_build,inp_quarry,inp_water], out_merged_3)
    arcpy.management.Dissolve(out_merged_3, out_dis_merged_3)
    
    arcpy.management.Merge([inp_build,inp_quarry,inp_water,inp_roads,inp_railways], out_merged_5)
    arcpy.management.Dissolve(out_merged_5, out_dis_merged_5)
    
    #print(inp_build, inp_quarry, inp_water, out_merged, out_dis_merged)

In [ ]:
# b) erst mergen dann maske abziehen, dann von gesamter diff maskieren

workspace = r"F:\newData\hesse_shp\grid"
merged_hesse3 = r"F:\newData\hesse_shp\c3merg_hes.shp"
dis_merged_hesse3 = r"F:\newData\hesse_shp\c3dismerg_hes.shp"

merge_list3 = []
for dirpath, dirnames, filenames in arcpy.da.Walk(workspace):
    for filename in filenames:
        if filename.startswith((r'buf10_dis_build', r'buf10_dis_water', r'buf10_dis_quarry')):
            merge_list3.append(r"{0}\{1}".format(dirpath, filename))

arcpy.management.Merge(merge_list3, merged_hesse3)
arcpy.management.Dissolve(merged_hesse3, dis_merged_hesse3)

In [ ]:
erased_frame3 = r"F:\newData\hesse_shp\erased3_hes.shp"
arcpy.analysis.Erase("bbmeged_hesse", dis_merged_hesse3, erased_frame3, None)

In [ ]:
diff_19_14 = r"F:\newData\testworkspace.gdb\copyraster1914"
diff_21_19 = r"F:\newData\testworkspace.gdb\copyraster2119_2"

diff_19_14_mask3 = r"F:\newData\diff19_14_hesse_mask3.tif"
diff_21_19_mask3 = r"F:\newData\diff21_19_hesse_mask3.tif"

outExtractByMask3_1914 = arcpy.sa.ExtractByMask(diff_19_14, erased_frame3)
outExtractByMask3_1914.save(diff_19_14_mask3)

outExtractByMask3_2119 = arcpy.sa.ExtractByMask(diff_21_19, erased_frame3)
outExtractByMask3_2119.save(diff_21_19_mask3)

In [ ]:
# b) erst mergen dann maske abziehen, dann von gesamter diff maskieren

workspace = r"F:\newData\hesse_shp\grid"
merged_hesse5 = r"F:\newData\hesse_shp\c5merg_hes.shp"
dis_merged_hesse5 = r"F:\newData\hesse_shp\c5dismerg_hes.shp"

merge_list5 = []
for dirpath, dirnames, filenames in arcpy.da.Walk(workspace):
    for filename in filenames:
        if filename.startswith((r'buf10_dis_build', r'buf10_dis_water', r'buf10_dis_quarry', r'buf10_dis_roads', r'buf10_dis_railw')):
            merge_list5.append(r"{0}\{1}".format(dirpath, filename))

arcpy.management.Merge(merge_list5, merged_hesse5)
arcpy.management.Dissolve(merged_hesse5, dis_merged_hesse5)

In [ ]:
erased_frame5 = r"F:\newData\hesse_shp\erased5_hes.shp"
arcpy.analysis.Erase("bbmeged_hesse", dis_merged_hesse5, erased_frame5, None)

In [ ]:
diff_19_14 = r"F:\newData\testworkspace.gdb\copyraster1914"
diff_21_19 = r"F:\newData\testworkspace.gdb\copyraster2119_2"

diff_19_14_mask5 = r"F:\newData\diff19_14_hesse_mask5.tif"
diff_21_19_mask5 = r"F:\newData\diff21_19_hesse_mask5.tif"

outExtractByMask5_1914 = arcpy.sa.ExtractByMask(diff_19_14, erased_frame5)
outExtractByMask5_1914.save(diff_19_14_mask5)

outExtractByMask5_2119 = arcpy.sa.ExtractByMask(diff_21_19, erased_frame5)
outExtractByMask5_2119.save(diff_21_19_mask5)

In [ ]:
# Erase with raster_50x50 to get mask

workspace = r"F:\newData\hesse_shp\grid"

i = 0
for dirpath, dirnames, filenames in arcpy.da.Walk(workspace):
    for filename in filenames:
        if filename.startswith(r'dis_hesse_raster_50x50'):
            inp_frame = r"{0}\{1}".format(dirpath, filename)
            #inp_frame = r"{0}".format(filename[:25])
            buffer_complete3 = r"F:\newData\hesse_shp\grid\osm_{}\buf10_dis_merged3_{}".format(i, i) 
            buffer_complete5 = r"F:\newData\hesse_shp\grid\osm_{}\buf10_dis_merged5_{}".format(i, i) 
            
            out_erased3 = r"F:\newData\hesse_shp\grid\osm_{}\buf10_erased3_{}".format(i, i)
            out_erased5 = r"F:\newData\hesse_shp\grid\osm_{}\buf10_erased5_{}".format(i, i)
            
            dis_filepath = r"{0}\osm_{1}\{2}_quarry_{3}".format(dirpath, i, filename[0:9], i)
            out_buf = r"{0}\osm_{1}\buf10_dis_quarry_{2}".format(dirpath, i, i)
            
            #arcpy.analysis.Erase(inp_frame, buffer_complete3, erased_frame, None)
            
            i = i+1